
# PREHEAT Retreat - Supervised Machine Learning 

#### Installing required R packages
These packages are already installed in the cloud.

if (!requireNamespace("readxl"))
  install.packages("readxl");
if (!requireNamespace("lubridate"))
  install.packages("lubridate");
if (!requireNamespace("tidyverse"))
  install.packages("tidyverse");
if (!requireNamespace("gtsummary"))
  install.packages("gtsummary");
if (!requireNamespace("flextable"))
  install.packages("flextable");
if (!requireNamespace("caret"))
  install.packages("caret");
if (!requireNamespace("randomForest"))
  install.packages("randomForest");
if (!requireNamespace("officer"))
  install.packages("officer");

#### Loading R packages required for this session

In [73]:
library(readxl);
library(lubridate);
library(tidyverse);
library(gtsummary);
library(flextable);
library(caret);
library(randomForest);
library(cardx);
library(officer);
library(MLmetrics);

#### Importing example dataset

In [74]:
# Load the data
arsenic_data <- data.frame(readxl::read_xlsx("../shared-data/data-in/Arsenic_data.xlsx"))

# View the top of the dataset
head(arsenic_data) 

,Well_ID,Water_Sample_Date,Casing_Depth,Well_Depth,Static_Water_Depth,Flow_Rate,pH,Detect_Concentration
,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>
1,W_1,9/24/12,52,165,41,60.0,7.7,ND
2,W_2,12/17/15,40,445,42,2.0,7.3,ND
3,W_3,2/2/15,45,160,40,40.0,7.4,ND
4,W_4,10/22/12,42,440,57,1.5,8.0,D
5,W_5,1/3/11,48,120,42,25.0,7.1,ND
6,W_6,12/15/15,60,280,32,10.0,8.2,D


The columns in this dataset are described below:

+ `Well_ID`: Unique id for each well (This is the sample identifier and not a predictive feature)
+ `Water_Sample_Date`: Date that the well was sampled 
+ `Casing_Depth`: Depth of the casing of the well (ft)
+ `Well_Depth`: Depth of the well (ft)
+ `Static_Water_Depth`: Static water depth in the well (ft)
+ `Flow_Rate`: Well flow rate (gallons per minute)
+ `pH`: pH of water sample
+ `Detect_Concentration`: Binary identifier (either non-detect "ND" or detect "D") if iAs concentration detected in water sample 

### Changing Data Types 
First, `Detect_Concentration` needs to be converted from a `character` to a `factor` so that Random Forest knows that the non-detect class is the baseline or "negative" class, while the detect class will be the "positive" class. `Water_Sample_Date` will be converted from a character to a date type using the `mdy()` function from the *lubridate* package. This is done so that the model understands this column contains dates.

In [75]:
# Convert Detect_Concentration from a character to a factor
# and set ND as the reference level and convert date
# https://dplyr.tidyverse.org/reference/mutate.html
arsenic_data <- dplyr::mutate(
  arsenic_data, #dataframe to mutate
  Detect_Concentration = relevel(factor(Detect_Concentration), ref = "ND"), #make Detect_Concentration a factor
  Water_Sample_Date = lubridate::mdy(Water_Sample_Date) #change data format
)

# Remove the Well_ID column
arsenic_data <- dplyr::select(arsenic_data, -Well_ID)

# Look at the first six rows
head(arsenic_data)

,Water_Sample_Date,Casing_Depth,Well_Depth,Static_Water_Depth,Flow_Rate,pH,Detect_Concentration
,<date>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,2012-09-24,52,165,41,60.0,7.7,ND
2,2015-12-17,40,445,42,2.0,7.3,ND
3,2015-02-02,45,160,40,40.0,7.4,ND
4,2012-10-22,42,440,57,1.5,8.0,D
5,2011-01-03,48,120,42,25.0,7.1,ND
6,2015-12-15,60,280,32,10.0,8.2,D


<br>

## Testing for Differences in Predictor Variables across the Outcome Classes

It is useful to run summary statistics on the variables that will be used as predictors in the algorithm to see if there are differences in distributions between the outcomes classes (either non-detect or detect in this case). Typically, greater significance often leads to better predictivity for a certain variable, since the model is better able to separate the classes. We'll use the `tbl_summary()` function from the *gtsummary* package. Note, this may only be practical with smaller datasets or for a subset of predictors if there are many.

For more information on the `tbl_summary()` function, check out this helpful [Tutorial](https://www.danieldsjoberg.com/gtsummary/articles/tbl_summary.html).

In [76]:
# Create the summary table
as_table <- gtsummary::tbl_summary(
  data = arsenic_data,
  by = Detect_Concentration,
  statistic = list(gtsummary::all_continuous() ~ "{mean} ({sd})")
)

# Add the sample size column
as_table <- gtsummary::add_n(as_table)

#Add p-values from a one-way ANOVA
as_table <- gtsummary::add_p(
  as_table,
  include = -Water_Sample_Date,
  test = list(gtsummary::all_continuous() ~ "oneway.test"),
  test.args = list(gtsummary::all_continuous() ~ list(var.equal = TRUE))
)

# Convert to a flextable
as_table <- gtsummary::as_flex_table(as_table)

# Bold the header
as_table <- flextable::bold(
  as_table,
  bold = TRUE,
  part = "header"
)

# Display in HTML
library(officer)
IRdisplay::display_html(officer::to_html(as_table))

Characteristic N ND N = 515 1 D N = 198 1 p-value 2 Water_Sample_Date 713 2013-06-05 (979.174260670888) 2013-03-05 (957.843005291701) Casing_Depth 713 74 (33) 55 (23) <0.001 Well_Depth 713 301 (144) 334 (128) 0.005 Static_Water_Depth 713 35 (12) 36 (13) 0.4 Flow_Rate 713 25 (33) 14 (16) <0.001 pH 713 7.45 (0.55) 7.82 (0.40) <0.001 1 Mean (SD) 2 One-way analysis of means

Note that N refers to the total sample number; ND refers to the samples that contained non-detectable levels of iAs; and D refers to the samples that contained detectable levels of iAs.

### Environmental Health Question 1
Which well water variables, spanning various geospatial locations, sampling dates, and well water attributes, significantly differ between samples containing detectable levels of iAs vs samples that are not contaminated/ non-detect?

<!-- :::answer
**Answer**: All of the evaluated descriptor variables are significantly different, with p<0.05 between detect and non-detect iAs samples, with the exception of the sample date and the static water depth.
::: -->

With these findings, we feel comfortable moving forward with these well water descriptive variables as predictors in our model.

<br>

### Setting up Cross Validation
At this point, we can move forward with training and testing a Random Forest model aimed at predicting whether or not detectable levels of iAs are present in well water samples. We'll take a glance at the distribution of `Detect_Concentration` between the two classes. 

In [77]:
# Set seed for reproducibility
set.seed(17)

# Establish a list of indices that will used to identify our training and testing data with a 60-40 split
tt_indices <- caret::createDataPartition(y = arsenic_data$Detect_Concentration, p = 0.6, list = FALSE)

##Caret package is widely used in R for machine learning.

In [78]:
# Use indices to make our training and testing datasets and view the number of Ds and NDs
iAs_train <- arsenic_data[tt_indices,]
table(iAs_train$Detect_Concentration)


 ND   D 
309 119 

In [79]:
iAs_test <- arsenic_data[-tt_indices,]
table(iAs_test$Detect_Concentration)


 ND   D 
206  79 

We can see that there are notably more non-detects (`ND`) than detects (`D`) in both our training and testing sets. This is something important to consider when evaluating our model's performance.

Now we can set up our cross validation and train our model. We will be using the `trainControl()` function from the *caret* package for this task. It is one of the most commonly used libraries for supervised machine learning in R and can be leveraged for a variety algorithms including RF, SVM, KNN, and others. This model will be trained with 5-fold cross validation. Additionally, we will test 2, 3, and 6 predictors through the `mtry` parameter.

See the *caret* documentation [here](https://cran.r-project.org/web/packages/caret/vignettes/caret.html).

In [80]:
#NOTE: We are using a Random Forest model. 
## y=f(x); where f=our Random Forest model, and x=our data.

# Establish the parameters for our cross validation with 5 folds
control <- caret::trainControl(method = 'cv',
                        number = 5,
                        search = 'grid',
                        classProbs = TRUE)

# Establish grid of predictors to test in our model as part of hyperparameter tuning
p <- ncol(arsenic_data) - 1 # p is the total number of predictors in the dataset

# We will test sqrt(p), p/2, and p predictors (2,3,& 6 predictors, respectively) to see which performs best
tunegrid_rf <- expand.grid(mtry = c(floor(sqrt(p)), p/2, p)) 

#Now we have our data and the inputs of our model ready!

<br>

## Predicting iAs Detection with a Random Forest (RF) Model

In [81]:
# Look at the column names in training dataset
colnames(iAs_train)

[1] "Water_Sample_Date"    "Casing_Depth"         "Well_Depth"          
[4] "Static_Water_Depth"   "Flow_Rate"            "pH"                  
[7] "Detect_Concentration"

In [82]:
# Train model
rf_train <- caret::train(x = iAs_train[,1:6], # Our predictor variables are in columns 1-6 of the dataframe
                         y = iAs_train[,7], # Our outcome variable is in column 7 of the dataframe
                         trControl = control, # Specify the cross-validation parameters we defined above
                         method = 'rf', # Specify we want to train a Random Forest
                         importance = TRUE, # This parameter calculates the variable importance for RF models specifically which can help with downstream analyses
                         tuneGrid = tunegrid_rf, # Specify the number of predictors we want to test as defined above
                         metric = "Accuracy", # Specify what evaluation metric we want to use to decide which model is the best
                  ) 

#Instead of "metric", Dr. Weigand prefers the term "loss function". 
#In machine learning, you always want to know (1) what model they're using, and (2) what is their loss function. 

In [83]:
# Look at the results of training
rf_train

#This tells us we built and trained 3 different RF models (based on mtry). 
##75% accuracy is not great, we would like it to be higher.

Random Forest 

428 samples
  6 predictor
  2 classes: 'ND', 'D' 

No pre-processing
Resampling: Cross-Validated (5 fold) 
Summary of sample sizes: 344, 342, 342, 342, 342 
Resampling results across tuning parameters:

  mtry  Accuracy   Kappa    
  2     0.7548726  0.3329518
  3     0.7361019  0.2891243
  6     0.7501661  0.3257679

Accuracy was used to select the optimal model using the largest value.
The final value used for the model was mtry = 2.

In [84]:
# Save the best model from our training. The best performing model is determined by the number of predictor variables we tested that resulted in the highest accuracy during the cross validation step.
rf_final <- rf_train$finalModel

In [85]:
# View confusion matrix for best model
rf_final

#Out of every 100 samples that are truly "D", the model incorrectly labels 61.3% of them as "ND" instead.
##This is a classic symptom of class imbalance. There are more ND cases than D cases in the training data, so the RF model learned that predicting "ND" is a good general strategy for minimizing overall error. It does great on the majority class but performs barely better than a coin flip on the minority class.


Call:
 randomForest(x = x, y = y, mtry = param$mtry, importance = TRUE) 
               Type of random forest: classification
                     Number of trees: 500
No. of variables tried at each split: 2

        OOB estimate of  error rate: 24.77%
Confusion matrix:
    ND  D class.error
ND 275 34   0.1100324
D   72 47   0.6050420

### Environmental Health Question 2
How can we train a random forest (RF) model to predict whether a well might be contaminated with iAs?


<!-- :::answer
**Answer**: As is standard practice with supervised ML, we split our full dataset into a training dataset and a test dataset using a 60-40 split. Using the *caret* package, we implemented 5-fold cross validation to train a RF while also testing different numbers of predictors to see which optimized performance. The model that resulted in the greatest accuracy was selected as the final model.
::: -->

Now we can see how well our model does on data it hasn't seen before by applying it to our testing data.

In [86]:
# Use our best model to predict the classes for our test data. We need to make sure we remove the column of Ds/NDs from our test data.
rf_res <- predict(rf_final, iAs_test %>% select(!Detect_Concentration))

# View a confusion matrix of the results and gauge model performance
# Be sure to include the 'positive' parameter to specify the correct positive class
confusionMatrix(rf_res, iAs_test$Detect_Concentration, positive = "D")

Confusion Matrix and Statistics

          Reference
Prediction  ND   D
        ND 178  47
        D   28  32
                                         
               Accuracy : 0.7368         
                 95% CI : (0.6817, 0.787)
    No Information Rate : 0.7228         
    P-Value [Acc > NIR] : 0.32445        
                                         
                  Kappa : 0.2907         
                                         
 Mcnemar's Test P-Value : 0.03767        
                                         
            Sensitivity : 0.4051         
            Specificity : 0.8641         
         Pos Pred Value : 0.5333         
         Neg Pred Value : 0.7911         
             Prevalence : 0.2772         
         Detection Rate : 0.1123         
   Detection Prevalence : 0.2105         
      Balanced Accuracy : 0.6346         
                                         
       'Positive' Class : D              
                                         

### Environmental Health Question 3
With this RF model, can we predict if iAs will be detected based on well water information? 

<!-- :::answer
**Answer**: We can use this model to predict if iAs can be detected in well water given that an overall accuracy of ~0.72 is decent, however we should consider other metrics that may influence how good we feel about this model depending on what is important to the question we are trying to answer. For example, the model did a good job at predicting non-detect data based on a sensitivity of ~0.85 and a NPV ~0.78, but struggled at predicting detect data based on a specificity of ~0.39 and a PPV of ~0.50. Additionally, the balanced accuracy of ~0.62 further emphasizes the difference in predictive ability of the model for non-detects and detects. If it is highly important to us that detects are classified correctly, we may want to improve this model before implementing it.
::: -->

## Class Imbalance

It is worth noting this discrepancy in predictive capabilities for detects vs. non-detects makes sense due to the observed class imbalance in our training data. There were notably more non-detects than detects in the training set, so the model was exposed to more of these data points and struggles to distinguish unique characteristics of detects when compared to non-detects. Additionally, we told the training algorithm to prioritize selecting a final model based on its overall accuracy. In the instances of a heavy class imbalance, it is common for a high accuracy to be achieved as the more prevalent class is predicted more often, though this doesn't give the full picture of the model's predictive capabilities. For example, if you consider a dog/cat case with a set of 90 dogs and 10 cats, a model could achieve 90% accuracy by predicting dog every time, which isn’t at all helpful in predicting cats.

This is particularly important, because for toxicology related datasets, the "positive" class often represents the class with greater public health risk/ interest but can have less data. For example, when you classify subjects based upon whether or not they have asthma based on gene expression data. Asthmatics would likely be the "positive" class, but given that asthmatics are less prevalent than non-asthmatics in the general population, they would likely represent the minority class too.  

To address this issue, a few methods can be considered. Full implementation of these approaches is beyond the scope of this workshop, but relevant resources for further exploration are given.

+ **Synthetic Minority Oversampling Technique (SMOTE)**- increases the number of minority classes in the training data, thereby reducing the class imbalance by synthetically generating additional samples derived from the existing minority class samples.
  + [SMOTE Oversampling & Tutorial On How To Implement In Python And R](https://spotintelligence.com/2023/02/17/smote-oversampling-python-r/#:~:text=Conclusion-,The%20SMOTE%20(Synthetic%20Minority%20Over%2Dsampling%20Technique)%20algorithm%20is,datasets%20that%20aren't%20balanced.)
  + [How to Use SMOTE for Imbalanced Data in R (With Example)](https://www.statology.org/smote-in-r/)

+ **Adjusting the loss function**- Loss functions in machine learning quantify the penalty for a bad prediction. They can be adjusted to where the minority class is penalized more forcing the model to learn to make fewer mistakes when predicting the minority class. 
  
+ **Alternative Performance Metrics**- When training the model, alternative metrics to overall accuracy may yield a more robust model capable of better predicting the minority class. Example alternatives may include balanced accuracy or an [F1-score](https://thedatascientist.com/f-1-measure-useful-imbalanced-class-problems/). The *caret* package further allows for [custom, user-defined metrics](https://topepo.github.io/caret/model-training-and-tuning.html#alternate-performance-metrics) to be evaluated during training by specifying the *summaryFunction* parameter in the `trainControl()` function, as seen below, in addition to the [`defaultSummary()` and `twoClassSummary()` functions](https://cran.r-project.org/web/packages/caret/vignettes/caret.html).

In the example code below, we're creating a function (`f1`) that will calculate the F1 score and find the optimal model with the highest F1 score as opposed to the highest accuracy as we did above. 

In [87]:
# Create f1 fuction
f1 <- function(data, lev = NULL, model = NULL) {
  # Creating a function to calculate the F1 score
  f1_val <- F1_Score(y_pred = data$pred, y_true = data$obs, positive = lev[1])
  c(F1 = f1_val)
}

# 5 fold CV
ctrl <- trainControl(
  method = "cv", 
  number = 5,
  classProbs = TRUE, 
  summaryFunction = f1
)

# Training the RF model
mod <- caret::train(x = iAs_train[,1:6], 
                         y = iAs_train[,7],
                         trControl = ctrl, 
                         method = 'rf', 
                         importance = TRUE, 
                         tuneGrid = tunegrid_rf, 
                         # Basing the best model performance off of the F1 score within 5 CV
                         metric = "F1")

In [88]:
# Look at the results of training
mod

Random Forest 

428 samples
  6 predictor
  2 classes: 'ND', 'D' 

No pre-processing
Resampling: Cross-Validated (5 fold) 
Summary of sample sizes: 343, 343, 342, 342, 342 
Resampling results across tuning parameters:

  mtry  F1       
  2     0.8413648
  3     0.8389455
  6     0.8387484

F1 was used to select the optimal model using the largest value.
The final value used for the model was mtry = 2.

In [89]:
# Save the best model from our training. The best performing model is determined by the number of predictor variables we tested that resulted in the highest accuracy during the cross validation step.
mod_final <- mod$finalModel

In [90]:
# View confusion matrix for best model
mod_final


Call:
 randomForest(x = x, y = y, mtry = param$mtry, importance = TRUE) 
               Type of random forest: classification
                     Number of trees: 500
No. of variables tried at each split: 2

        OOB estimate of  error rate: 25%
Confusion matrix:
    ND  D class.error
ND 275 34   0.1100324
D   73 46   0.6134454

In [91]:
# Use our best model to predict the classes for our test data. We need to make sure we remove the column of Ds/NDs from our test data.
mod_res <- predict(mod_final, iAs_test %>% select(!Detect_Concentration))

# View a confusion matrix of the results and gauge model performance
# Be sure to include the 'positive' parameter to specify the correct positive class
confusionMatrix(mod_res, iAs_test$Detect_Concentration, positive = "D")

Confusion Matrix and Statistics

          Reference
Prediction  ND   D
        ND 177  48
        D   29  31
                                          
               Accuracy : 0.7298          
                 95% CI : (0.6743, 0.7805)
    No Information Rate : 0.7228          
    P-Value [Acc > NIR] : 0.42506         
                                          
                  Kappa : 0.2718          
                                          
 Mcnemar's Test P-Value : 0.04024         
                                          
            Sensitivity : 0.3924          
            Specificity : 0.8592          
         Pos Pred Value : 0.5167          
         Neg Pred Value : 0.7867          
             Prevalence : 0.2772          
         Detection Rate : 0.1088          
   Detection Prevalence : 0.2105          
      Balanced Accuracy : 0.6258          
                                          
       'Positive' Class : D               
                              

For more in-depth information and additional ways to address class imbalance check out [How to Deal with Imbalanced Data in Classification](https://medium.com/game-of-bits/how-to-deal-with-imbalanced-data-in-classification-bd03cfc66066).

### Answer to Environmental Health Question 4
How could this RF model be improved upon, acknowledging that there is class imbalance?

<!-- :::answer
**Answer**: We can implement SMOTE to increase the number of training data points for the minority class thereby reducing the class imbalance. In conjunction with using SMOTE, another approach includes selecting an alternative performance metric during training that does a better job taking the existing class imbalance into consideration, such as balanced accuracy or an F1-score, improves our predictive ability for the minority class.
::: -->


## Concluding Remarks

In conclusion, this training module has provided an introduction to supervised machine learning using classification techniques in R. Machine learning is a powerful tool that can help researchers gain new insights and improve models to analyze complex datasets faster and in a more comprehensive way. The example we've explored demonstrates the utility of supervised machine learning models on an environmentally relevant dataset.

### Additional Resources
To learn more check out the following resources: 

+ [IBM - What is Machine Learning](https://www.ibm.com/topics/machine-learning)
+ [Curate List of AI and Machine Learning Resources](https://medium.com/machine-learning-in-practice/my-curated-list-of-ai-and-machine-learning-resources-from-around-the-web-9a97823b8524)
+ [Introduction to Machine Learning in R](https://machinelearningmastery.com/machine-learning-in-r-step-by-step/)
+ Machine Learning by Mueller, J. P. (2021). Machine learning for dummies. John Wiley &amp; Sons.


### Test Your Knowledge 

Using the "Manganese_data.xlsx", use RF to determine if well water data can be accurate predictors of Manganese detection. The data is structured similarly to the Arsenic data used in this module, however it now includes 4 additional features:

+ `Longitude`: Longitude of address (decimal degrees)
+ `Latitude`: Latitude of address (decimal degrees)
+ `Stream_Distance`: Euclidean distance to the nearest stream (feet)
+ `Elevation`: Surface elevation of the sample location (feet)